# CorGAN Synthetic Data Generation for MIMIC-III

**Last updated:** 2026-02-28

This notebook trains a [CorGAN](https://doi.org/10.1093/jamia/ocz120) model on MIMIC-III diagnosis codes and generates synthetic electronic health records (EHRs).

### What You'll Need
- **MIMIC-III access** via [PhysioNet](https://physionet.org/content/mimiciii/) — or use **demo mode** (no data required)
- **Google Colab** with GPU runtime (Runtime > Change runtime type > T4 GPU), or a local machine with PyTorch
- **Time:** Demo ~20–30 min on T4 | Production ~2–4 hrs on T4

### What You'll Get
- A trained CorGAN model checkpoint
- Synthetic MIMIC-III patients (default: 1,000 in demo, 10,000 in production)
- A quality report (`quality_report.json`) with evaluation metrics

### How It Works (6 Steps)
1. **Setup** — Install PyHealth, detect GPU
2. **Configure** — Set training parameters (demo vs production)
3. **Upload Data** — Upload MIMIC-III CSVs or use demo data
4. **Train** — Pre-train autoencoder, then adversarial WGAN training
5. **Generate** — Synthesize patient records
6. **Evaluate** — Compare real vs synthetic data quality

### What Makes CorGAN Different

CorGAN uses a **CNN autoencoder** combined with a **Wasserstein GAN (WGAN)** to generate synthetic patient records as flat bags-of-codes (binary vectors of ICD-9 diagnoses).

The key insight is that 1D convolutions in the autoencoder capture **inter-code correlations** — for example, that diabetes and hypertension frequently co-occur — which plain linear autoencoders miss. The three components work together:

- **Autoencoder**: Learns a compressed latent representation of multi-hot ICD-9 code vectors. The CNN layers capture which codes tend to appear together.
- **Generator**: Produces synthetic latent codes from random noise.
- **Discriminator (Critic)**: Distinguishes real latent codes from generated ones, providing the adversarial training signal.

Training proceeds in two phases: (1) the autoencoder is pre-trained to learn good latent representations, then (2) the WGAN trains the generator to produce realistic latent codes that the decoder maps back to binary code vectors.

Unlike HALO (which generates sequential visits via a Transformer), CorGAN aggregates all of a patient's diagnoses across admissions into a single flat set.

### Reference
Baowaly et al., "Synthesizing Electronic Health Records Using Improved Generative Adversarial Networks", *JAMIA*, 2019. [DOI: 10.1093/jamia/ocz120](https://doi.org/10.1093/jamia/ocz120)

---

> **Colab timeout warning:** Free Colab sessions disconnect after ~90 min of inactivity. For production training, consider [Colab Pro](https://colab.research.google.com/signup) or running on a local GPU/SLURM cluster.

## 1. Setup & Installation

In [ ]:
import subprocess, sys

FORK = "jalengg"
BRANCH = "corgan-pr-integration"
install_url = f"git+https://github.com/{FORK}/PyHealth.git@{BRANCH}"

# Only install from GitHub in Colab; locally, use the editable install
try:
    import google.colab
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", install_url, "--quiet", "--no-cache-dir"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("PyHealth installation failed")
    print(f"PyHealth installed from {FORK}/{BRANCH}")
except ImportError:
    print("Not in Colab — using local PyHealth installation")

In [ ]:
import os, json, random, time
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Environment detection
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# GPU detection
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    print("WARNING: No GPU detected. Training will be slow.")
    DEVICE = "cpu"

print(f"PyTorch {torch.__version__} | Environment: {'Colab' if IN_COLAB else 'Local'}")

## 2. Configuration

All hyperparameters are centralized here. Change `PRESET` to switch between a quick demo and full production training. You should not need to modify any other cell.

In [ ]:
# ============================================================
# CONFIGURATION — Modify these parameters
# ============================================================

PRESET = "demo"  # "demo" or "production"

# Training parameters
if PRESET == "demo":
    EPOCHS = 5              # Quick smoke test (~20-30 min on T4)
    BATCH_SIZE = 64
    N_SYNTHETIC_SAMPLES = 1000
    N_EPOCHS_PRETRAIN = 1   # Autoencoder pre-training epochs
elif PRESET == "production":
    EPOCHS = 50             # Full training (~2-4 hrs on T4)
    BATCH_SIZE = 512
    N_SYNTHETIC_SAMPLES = 10000
    N_EPOCHS_PRETRAIN = 3

# Model architecture
LATENT_DIM = 128            # Generator + decoder latent space dimension
HIDDEN_DIM = 128            # Generator MLP hidden layer width
AUTOENCODER_TYPE = "cnn"    # "cnn", "cnn8layer", or "linear"

# WGAN parameters
# N_ITER_D: Discriminator updates per generator step. Higher = stronger critic
# but slower training. 5 is the WGAN default from Arjovsky et al.
N_ITER_D = 5
# Weight clipping enforces the Lipschitz constraint on the discriminator.
# Smaller range = more stable but slower convergence.
CLAMP_LOWER = -0.01
CLAMP_UPPER = 0.01
LR = 0.001                  # Learning rate for all optimizers

# Reproducibility
SEED = 42

# Paths
BASE_DIR = "/content/drive/MyDrive/CorGAN_Training" if IN_COLAB else "./corgan_training"
DATA_DIR = os.path.join(BASE_DIR, "data")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

print(f"Preset: {PRESET}")
print(f"Training: {EPOCHS} epochs, batch size {BATCH_SIZE}")
print(f"Architecture: {AUTOENCODER_TYPE} autoencoder, latent_dim={LATENT_DIM}")
print(f"Output: {N_SYNTHETIC_SAMPLES} synthetic patients")
print(f"Base directory: {BASE_DIR}")

## 3. Data Upload

**Two paths:**
- **With MIMIC-III:** Upload `DIAGNOSES_ICD.csv` (or `.csv.gz`) from your PhysioNet download, plus `ADMISSIONS.csv`
- **Demo mode (no MIMIC-III):** A synthetic dataset is generated automatically so you can run the full pipeline

If you've already uploaded files to Google Drive in a previous session, they'll be detected automatically.

In [ ]:
# Mount Google Drive (Colab only) and create directories
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")

for d in [DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"Directory ready: {d}")

In [ ]:
from pyhealth.datasets.sample_dataset import InMemorySampleDataset

MIMIC3_ROOT = None  # Set to your MIMIC-III directory path, or leave None for demo

# Try to detect MIMIC-III data
if MIMIC3_ROOT and os.path.isdir(MIMIC3_ROOT):
    USE_DEMO = False
    print(f"Using MIMIC-III data from: {MIMIC3_ROOT}")
elif IN_COLAB:
    # Try uploading files
    print("Upload DIAGNOSES_ICD.csv (or .csv.gz) and ADMISSIONS.csv from MIMIC-III:")
    try:
        uploaded = files.upload()
        if uploaded:
            for fname, content in uploaded.items():
                dest = os.path.join(DATA_DIR, fname)
                with open(dest, "wb") as f:
                    f.write(content)
                print(f"Saved: {dest} ({len(content):,} bytes)")
            MIMIC3_ROOT = DATA_DIR
            USE_DEMO = False
        else:
            USE_DEMO = True
    except Exception:
        USE_DEMO = True
else:
    USE_DEMO = True

if USE_DEMO:
    print("MIMIC-III not found \u2014 using synthetic demo data")
    print("Demo mode uses a small random vocabulary to demonstrate the full pipeline.")

    # Generate demo data: 200 patients with random ICD-9-like codes
    demo_codes = [f"D{i:03d}" for i in range(50)]  # 50-code vocabulary
    rng = np.random.RandomState(SEED)
    demo_samples = []
    for pid in range(200):
        n_codes = rng.randint(3, 20)
        codes = rng.choice(demo_codes, size=n_codes, replace=False).tolist()
        demo_samples.append({"patient_id": f"demo_{pid}", "visits": codes})

    dataset = InMemorySampleDataset(
        samples=demo_samples,
        input_schema={"visits": "multi_hot"},
        output_schema={},
        dataset_name="demo",
        task_name="CorGANGeneration",
    )
    print(f"Demo dataset: {len(dataset)} patients, {len(demo_codes)} unique codes")

In [ ]:
if not USE_DEMO:
    from pyhealth.datasets import MIMIC3Dataset
    from pyhealth.tasks.corgan_generation import CorGANGenerationMIMIC3

    mimic3_dataset = MIMIC3Dataset(
        root=MIMIC3_ROOT,
        tables=["diagnoses_icd"],
        code_mapping={},
        refresh_cache=False,
    )

    task = CorGANGenerationMIMIC3()
    dataset = mimic3_dataset.set_task(task)
    print(f"MIMIC-III dataset: {len(dataset)} patients")

# Display sample
print(f"\nSample patient: {dataset[0]}")

## 4. Training

CorGAN training has **two phases**:

1. **Autoencoder pre-training**: The CNN autoencoder learns to compress multi-hot ICD-9 code vectors into a low-dimensional latent space and reconstruct them. This teaches the model which codes tend to co-occur.

2. **WGAN adversarial training**: The generator learns to produce latent codes that the decoder maps back to realistic binary code vectors. The discriminator (critic) is trained to distinguish real from generated latent codes, providing the Wasserstein distance as a training signal.

After training, three loss curves are plotted: autoencoder reconstruction loss, discriminator Wasserstein distance, and generator loss. Stabilization of the Wasserstein distance indicates convergence.

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
print(f"Random seed set to {SEED}")

In [ ]:
from pyhealth.datasets import split_by_patient

# Split dataset
train_dataset, val_dataset, test_dataset = split_by_patient(
    dataset, ratios=[0.85, 0.1, 0.05], seed=SEED
)
print(f"Split: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

# Save hyperparameter config
config_record = {
    "preset": PRESET,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "n_epochs_pretrain": N_EPOCHS_PRETRAIN,
    "latent_dim": LATENT_DIM,
    "hidden_dim": HIDDEN_DIM,
    "autoencoder_type": AUTOENCODER_TYPE,
    "n_iter_d": N_ITER_D,
    "clamp_lower": CLAMP_LOWER,
    "clamp_upper": CLAMP_UPPER,
    "lr": LR,
    "seed": SEED,
    "n_synthetic_samples": N_SYNTHETIC_SAMPLES,
    "use_demo": USE_DEMO,
    "timestamp": datetime.now().isoformat(),
}
config_path = os.path.join(CHECKPOINT_DIR, "config.json")
with open(config_path, "w") as f:
    json.dump(config_record, f, indent=2)
print(f"Config saved to {config_path}")

In [ ]:
from pyhealth.models.generators.corgan import CorGAN

model = CorGAN(
    dataset=train_dataset,
    latent_dim=LATENT_DIM,
    hidden_dim=HIDDEN_DIM,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    n_epochs_pretrain=N_EPOCHS_PRETRAIN,
    n_iter_D=N_ITER_D,
    clamp_lower=CLAMP_LOWER,
    clamp_upper=CLAMP_UPPER,
    lr=LR,
    autoencoder_type=AUTOENCODER_TYPE,
    save_dir=CHECKPOINT_DIR,
)

print(f"Model initialized: input_dim={model.input_dim}, autoencoder={AUTOENCODER_TYPE}")
print(f"Training: {EPOCHS} adversarial epochs + {N_EPOCHS_PRETRAIN} pretrain epochs\n")

history = model.train_model(train_dataset)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Autoencoder loss
axes[0].plot(history["autoencoder_loss"], marker="o", color="tab:blue")
axes[0].set_title("Autoencoder Loss (Pre-training)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Reconstruction Loss")
axes[0].grid(True, alpha=0.3)

# Discriminator loss (Wasserstein distance)
axes[1].plot(history["discriminator_loss"], marker="o", color="tab:orange")
axes[1].set_title("Discriminator Loss (Wasserstein Distance)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("W-Distance")
axes[1].grid(True, alpha=0.3)

# Generator loss
axes[2].plot(history["generator_loss"], marker="o", color="tab:green")
axes[2].set_title("Generator Loss")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Loss")
axes[2].grid(True, alpha=0.3)

plt.suptitle("CorGAN Training Loss Curves", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Loss curves saved to output/loss_curves.png")

## 5. Generation

CorGAN generates a **flat bag-of-codes** per patient — a single set of ICD-9 diagnosis codes representing all their diagnoses across admissions. Unlike HALO (which generates sequential visits), there is no `VISIT_NUM` column in the output.

In [ ]:
print(f"Generating {N_SYNTHETIC_SAMPLES} synthetic patients...")
synthetic = model.synthesize_dataset(num_samples=N_SYNTHETIC_SAMPLES)
print(f"Generated {len(synthetic)} patients")

# Display sample
sample_df = pd.DataFrame([
    {
        "patient_id": p["patient_id"],
        "n_codes": len(p["visits"]),
        "sample_codes": ", ".join(p["visits"][:5]) + ("..." if len(p["visits"]) > 5 else ""),
    }
    for p in synthetic[:10]
])
display(sample_df)

In [ ]:
# Save as JSON
json_path = os.path.join(OUTPUT_DIR, "synthetic_patients.json")
with open(json_path, "w") as f:
    json.dump(synthetic, f, indent=2)
print(f"JSON saved: {json_path} ({os.path.getsize(json_path):,} bytes)")

# Save as CSV (flat: SUBJECT_ID, ICD9_CODE)
rows = []
for p in synthetic:
    for code in p["visits"]:
        rows.append({"SUBJECT_ID": p["patient_id"], "ICD9_CODE": code})
csv_df = pd.DataFrame(rows)
csv_path = os.path.join(OUTPUT_DIR, "synthetic_patients.csv")
csv_df.to_csv(csv_path, index=False)
print(f"CSV saved: {csv_path} ({len(csv_df)} rows, {os.path.getsize(csv_path):,} bytes)")

display(csv_df.head(10))

## 6. Results & Evaluation

This section validates synthetic data quality by comparing it against the real training data. CorGAN's core contribution is capturing inter-code correlations, so we measure code frequency fidelity (Pearson correlation) as the central metric.

In [ ]:
# 6a. Vocabulary Coverage
all_generated_codes = set(code for p in synthetic for code in p["visits"])
vocab_size = train_dataset.input_processors["visits"].size()
coverage = len(all_generated_codes) / vocab_size * 100
print(f"Unique codes generated: {len(all_generated_codes)}")
print(f"Vocabulary size: {vocab_size}")
print(f"Vocabulary coverage: {coverage:.1f}%")
if coverage < 30:
    print("WARNING: Low coverage may indicate mode collapse.")

In [ ]:
# 6b. Code Count Statistics
synth_code_counts = [len(p["visits"]) for p in synthetic]

# Real training data code counts
real_code_counts = []
for i in range(len(train_dataset)):
    sample = train_dataset[i]
    n_codes = int(sample["visits"].sum().item())
    real_code_counts.append(n_codes)

print("=== Codes per Patient ===")
print(f"{'':>20s} {'Real':>10s} {'Synthetic':>10s}")
print(f"{'Mean':>20s} {np.mean(real_code_counts):>10.1f} {np.mean(synth_code_counts):>10.1f}")
print(f"{'Std':>20s} {np.std(real_code_counts):>10.1f} {np.std(synth_code_counts):>10.1f}")
print(f"{'Min':>20s} {np.min(real_code_counts):>10d} {np.min(synth_code_counts):>10d}")
print(f"{'Max':>20s} {np.max(real_code_counts):>10d} {np.max(synth_code_counts):>10d}")
print(f"{'Median':>20s} {np.median(real_code_counts):>10.1f} {np.median(synth_code_counts):>10.1f}")

# Histogram comparison
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, max(max(real_code_counts), max(synth_code_counts)), 30)
ax.hist(real_code_counts, bins=bins, alpha=0.6, label="Real", density=True, color="tab:blue")
ax.hist(synth_code_counts, bins=bins, alpha=0.6, label="Synthetic", density=True, color="tab:orange")
ax.set_xlabel("Number of Codes per Patient")
ax.set_ylabel("Density")
ax.set_title("Code Count Distribution: Real vs Synthetic")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "code_count_histogram.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 6c. Code Frequency Comparison
from collections import Counter

# Build index-to-code mapping from the processor's label_vocab
processor = train_dataset.input_processors["visits"]
idx_to_code = {idx: code for code, idx in processor.label_vocab.items()}

# Count code frequencies in real data
real_freq = Counter()
for i in range(len(train_dataset)):
    vec = train_dataset[i]["visits"]
    active = torch.where(vec > 0)[0].tolist()
    for idx in active:
        code = idx_to_code.get(idx)
        if code and code not in ("<pad>", "<unk>"):
            real_freq[code] += 1

# Count code frequencies in synthetic data
synth_freq = Counter()
for p in synthetic:
    for code in p["visits"]:
        synth_freq[code] += 1

# Align frequencies on common codes
all_codes = sorted(set(real_freq.keys()) | set(synth_freq.keys()))
real_vals = np.array([real_freq.get(c, 0) for c in all_codes], dtype=float)
synth_vals = np.array([synth_freq.get(c, 0) for c in all_codes], dtype=float)

# Normalize to frequencies
if real_vals.sum() > 0:
    real_vals /= real_vals.sum()
if synth_vals.sum() > 0:
    synth_vals /= synth_vals.sum()

# Pearson correlation
pearson_r = np.corrcoef(real_vals, synth_vals)[0, 1] if len(all_codes) > 1 else 0.0
print(f"Code frequency Pearson r = {pearson_r:.4f}")

# Plot top 20 codes
n_top = min(20, len(all_codes))
top_codes_idx = np.argsort(real_vals)[-n_top:][::-1]
top_codes = [all_codes[i] for i in top_codes_idx]
top_real = [real_vals[i] for i in top_codes_idx]
top_synth = [synth_vals[i] for i in top_codes_idx]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(top_codes))
width = 0.35
ax.bar(x - width / 2, top_real, width, label="Real", color="tab:blue", alpha=0.8)
ax.bar(x + width / 2, top_synth, width, label="Synthetic", color="tab:orange", alpha=0.8)
ax.set_xlabel("ICD-9 Code")
ax.set_ylabel("Frequency (normalized)")
ax.set_title(f"Top {n_top} Code Frequencies: Real vs Synthetic (Pearson r = {pearson_r:.3f})")
ax.set_xticks(x)
ax.set_xticklabels(top_codes, rotation=45, ha="right", fontsize=8)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "code_frequency_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 6d. All-Zeros Detection
empty_patients = sum(1 for p in synthetic if len(p["visits"]) == 0)
print(f"Empty patients (all-zeros): {empty_patients} / {len(synthetic)}")
if empty_patients > 0:
    print("WARNING: Some patients have no codes (all-zeros generation).")
    print("Consider: adjusting binarization threshold, retraining with more epochs,")
    print("or reducing learning rate.")
else:
    print("No all-zeros patients detected.")

In [ ]:
# 6e. Quality Report
quality_report = {
    "total_synthetic_patients": len(synthetic),
    "mean_codes_per_patient": float(np.mean(synth_code_counts)),
    "std_codes_per_patient": float(np.std(synth_code_counts)),
    "min_codes": int(np.min(synth_code_counts)),
    "max_codes": int(np.max(synth_code_counts)),
    "unique_codes_generated": len(all_generated_codes),
    "vocabulary_size": vocab_size,
    "vocabulary_coverage_percent": round(coverage, 2),
    "empty_patients_count": empty_patients,
    "code_frequency_pearson_r": round(float(pearson_r), 4),
    "seed": SEED,
    "preset": PRESET,
    "epochs": EPOCHS,
    "timestamp": datetime.now().isoformat(),
}

report_path = os.path.join(OUTPUT_DIR, "quality_report.json")
with open(report_path, "w") as f:
    json.dump(quality_report, f, indent=2)

print(f"Quality report saved to {report_path}")
print(json.dumps(quality_report, indent=2))

## 7. Download & Next Steps

In [ ]:
# Download outputs
print("=== Output Files ===")
for fname in ["synthetic_patients.json", "synthetic_patients.csv", "quality_report.json"]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        print(f"  {fname}: {size:,} bytes")

if IN_COLAB:
    for fname in ["synthetic_patients.csv", "quality_report.json", "synthetic_patients.json"]:
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.exists(fpath):
            files.download(fpath)
    print("\nFiles downloaded! Also backed up in Google Drive.")
else:
    print(f"\nFiles saved in: {OUTPUT_DIR}")

## Congratulations!

You've trained a CorGAN model and generated synthetic MIMIC-III patient records.

### Next Steps
- **Use your synthetic data**: Load `synthetic_patients.json` into downstream PyHealth pipelines
- **Generate more patients**: Re-run Section 5 with a larger `N_SYNTHETIC_SAMPLES`
- **Production training**: Change `PRESET = "production"` and re-run from Section 2

### Troubleshooting

**Out of memory (OOM)**: Reduce `BATCH_SIZE` (try 32 or 16)

**Mode collapse** (many empty patients or low vocabulary coverage):
- Check `quality_report.json` for `empty_patients_count` and `vocabulary_coverage_percent`
- Try reducing `LR` (e.g., 0.0001)
- Increase `N_ITER_D` (e.g., 10) to strengthen the critic
- Train for more epochs

**Slow training**: Use GPU runtime (Runtime > Change runtime type > T4 GPU)

### References
- [CorGAN Paper (Baowaly et al., JAMIA 2019)](https://doi.org/10.1093/jamia/ocz120)
- [PyHealth Documentation](https://pyhealth.readthedocs.io/)
- [MIMIC-III on PhysioNet](https://physionet.org/content/mimiciii/)